<a href="https://colab.research.google.com/github/DSC-SPIDAL/DeepKG/blob/main/July5ColabCheck.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Check runs

In [5]:
# ==========================================
# 3. COMPARE TO GOLDENKB (STRICT SCHEMA MERGE)
# ==========================================
if GOLDENKB_ID and yield_report:
    print("\n🔗 Loading GOLDENKB (Tab: 'Datasets') from Google Sheets...")
    def normalize_text(text):
        """Strict normalization to prevent false-positive 'conflicts' over spacing/case."""
        if pd.isna(text): return ""
        return re.sub(r'\s+', ' ', str(text).lower().strip())

    try:
        # Open the spreadsheet strictly to the 'Datasets' tab
        spreadsheet = gc.open_by_key(GOLDENKB_ID)
        worksheet = spreadsheet.worksheet('Datasets')
        print(f"✅ Successfully connected to tab: '{worksheet.title}'")

        rows = worksheet.get_all_values()
        if not rows:
            raise ValueError("The GoldenKB sheet is completely empty.")

        golden_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])

        # 💡 SCHEMA MAPPING: Translates LLM Markdown headers to your Strict GitHub Schema
        SCHEMA_MAP = {
            "Dataset Name": "Variant Name",
            "Detailed Description": "Description",
            "Time interval between points": "Frequency",
            "Number of Time Points": "Num Time Points",
            "Number of Locations/Series": "Num Locations/Series",
            "Primary Source Repository": "Primary Creator",
            "Canonical Name": "Canonical Name",
            "Aliases": "Aliases",
            "Type": "Type",
            "Total Variables": "Total Variables",
            "Variables per Location": "Variables per Location",
            "Domain": "Domain",
            "Primary URL": "Primary URL",
            "Link to Data (Actual Source)": "Link to Data (Actual Source)",
            "Other URL": "Other URL"
        }

        # Filter out metadata and Confidence (C) columns from GoldenKB so we only compare core facts
        kb_core_cols = [c for c in golden_df.columns if not c.endswith('(C)') and c not in
                        ['DatasetID', 'Job_Created', 'Date_Created', 'Project_Created',
                         'Job_Updated', 'Date_Updated', 'Project_Updated', 'Overall Confidence', 'License']]

        golden_core = golden_df[kb_core_cols].copy()

        # 1. MELT THE GOLDEN KB INTO CELL-BY-CELL FORMAT
        # We transform the "wide" schema into [Variant Name, Entity_Name (Column), Value]
        golden_melted = golden_core.melt(id_vars=['Variant Name'], var_name='Entity_Name', value_name='Value_Golden')
        golden_melted['Norm_Value_Golden'] = golden_melted['Value_Golden'].apply(normalize_text)
        golden_melted['Entity_Name_Norm'] = golden_melted['Entity_Name'].str.lower().str.strip()

        # Remove empty cells from the GoldenKB so we don't flag them as false conflicts
        golden_melted = golden_melted[golden_melted['Norm_Value_Golden'] != ""]

        # 2. ALIGN AND MELT THE AMNESIA RUNS
        all_extractions = []
        invalid_vals = {'[missing]', '[skipped]', '', 'n/a', 'nan'}

        for proj, df in project_dfs.items():
            # Apply the schema mapping to the LLM outputs
            df_mapped = df.rename(columns=SCHEMA_MAP)

            if 'Variant Name' not in df_mapped.columns:
                continue

            # Keep only columns that exist in the GoldenKB Schema
            valid_cols = [c for c in df_mapped.columns if c in kb_core_cols]

            # Melt to match Golden KB shape
            melted = df_mapped[valid_cols].melt(id_vars=['Variant Name'], var_name='Entity_Name', value_name='Value_Model')
            melted['Project'] = proj

            # Filter out [missing]/[Skipped] placeholders
            melted = melted[~melted['Value_Model'].str.lower().isin(invalid_vals)]
            all_extractions.append(melted)

        if all_extractions:
            df_models = pd.concat(all_extractions, ignore_index=True)
            df_models['Norm_Value_Model'] = df_models['Value_Model'].apply(normalize_text)
            df_models['Entity_Name_Norm'] = df_models['Entity_Name'].str.lower().str.strip()

            # 3. STRICT MERGE
            merged = pd.merge(
                df_models,
                golden_melted,
                on=['Variant Name', 'Entity_Name_Norm'],
                how='outer',
                indicator=True,
                suffixes=('_Model', '_Golden')
            )

            # 4. CATEGORIZE RESULTS
            matches = merged[(merged['_merge'] == 'both') & (merged['Norm_Value_Model'] == merged['Norm_Value_Golden'])]
            conflicts = merged[(merged['_merge'] == 'both') & (merged['Norm_Value_Model'] != merged['Norm_Value_Golden'])]
            novel = merged[merged['_merge'] == 'left_only']

            print("\n==================================================")
            print(" 🏆 AMNESIA RUNS vs GOLDENKB CONSENSUS")
            print("==================================================")
            print(f"✅ Exact Matches (True Positives): {len(matches)}")
            print(f"❌ Conflicts (Requires Human Review): {len(conflicts)}")
            print(f"🌟 Novel Discoveries (Pending Merge): {len(novel)}")

            # 5. EXPORT ACTIONABLE REPORTS
            novel_path = "Novel_Discoveries_Merge.csv"
            conflict_path = "Conflicts_Pending_Repair.csv"

            entity_col = 'Entity_Name_Model' if 'Entity_Name_Model' in novel.columns else 'Entity_Name'

            if not novel.empty:
                novel_out = novel[['Project', 'Variant Name', entity_col, 'Value_Model']].rename(columns={entity_col: 'Schema_Column'})
                novel_out.to_csv(novel_path, index=False)
                print(f"\n💾 Saved {len(novel_out)} Novel Discoveries to: /content/{novel_path}")

            if not conflicts.empty:
                conflict_cols = ['Project', 'Variant Name', entity_col, 'Value_Model', 'Value_Golden']
                conflicts_out = conflicts[[c for c in conflict_cols if c in conflicts.columns]].rename(columns={entity_col: 'Schema_Column'})
                conflicts_out.to_csv(conflict_path, index=False)
                print(f"💾 Saved {len(conflicts_out)} Conflicts to: /content/{conflict_path}")

            print("\n-> Click the 📁 Folder icon on the left panel in Colab to download the CSVs!")

    except Exception as e:
        print(f"\n⚠️ Strict Evaluation Failed: {e}")


🔗 Loading GOLDENKB (Tab: 'Datasets') from Google Sheets...
✅ Successfully connected to tab: 'Datasets'

 🏆 AMNESIA RUNS vs GOLDENKB CONSENSUS
✅ Exact Matches (True Positives): 880
❌ Conflicts (Requires Human Review): 976
🌟 Novel Discoveries (Pending Merge): 1376

💾 Saved 1376 Novel Discoveries to: /content/Novel_Discoveries_Merge.csv
💾 Saved 976 Conflicts to: /content/Conflicts_Pending_Repair.csv

-> Click the 📁 Folder icon on the left panel in Colab to download the CSVs!


#### Cell 1: Set up Environment

In [ ]:
%cd /content
!rm -rf DeepKG
!git clone https://github.com/DSC-SPIDAL/DeepKG.git > /dev/null 2>&1

!pip uninstall -y google-generativeai llama-index-llms-gemini llama-index-embeddings-gemini > /dev/null 2>&1
!pip install -qU "google-genai>=2.0.0" llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-retrievers-bm25 gspread bm25s python-dotenv nest_asyncio arxiv pypdf > /dev/null 2>&1
print("✅ Installed. Please Restart Session.")

/content
✅ Installed. Please Restart Session.


#### Cell 2: Authenticate

In [ ]:
import os
from google.colab import auth, userdata, drive

print("🔑 Authenticating & Mounting Drive...")
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)

api_key = userdata.get('GEMINI_API_KEY')
os.environ["GEMINI_API_KEY"] = api_key
if "GOOGLE_API_KEY" in os.environ:
    del os.environ["GOOGLE_API_KEY"]

%cd /content/DeepKG
print("✅ Ready.")

🔑 Authenticating & Mounting Drive...
Mounted at /content/drive
/content/DeepKG
✅ Ready.


#### 3:Execute

In [ ]:
# ==========================================
# CELL 3: RUN THE TRUE CLOUD BASELINE (Anti-Recursion V3)
# ==========================================
import os
import glob
import warnings
import nest_asyncio
import re
import io
import time
import pandas as pd
import requests

warnings.filterwarnings("ignore", category=FutureWarning)
nest_asyncio.apply()

from google.colab import output, userdata, auth
import gspread
from google.auth import default
output.no_vertical_scroll()

# ---------------------------------------------------------
# 🔥 HOT-PATCH 1: Rewrite DeepCollector Imports dynamically
# ---------------------------------------------------------
print("🔧 Patching DeepCollector for the new google-genai SDK...")
for filepath in glob.glob("/content/DeepKG/**/*.py", recursive=True):
    with open(filepath, "r") as f: content = f.read()
    new_content = content.replace("llama_index.llms.gemini", "llama_index.llms.google_genai")
    new_content = new_content.replace("from llama_index.llms.google_genai import Gemini", "from llama_index.llms.google_genai import GoogleGenAI as Gemini")
    new_content = new_content.replace("llama_index.embeddings.gemini", "llama_index.embeddings.google_genai")
    new_content = new_content.replace("from llama_index.embeddings.google_genai import GeminiEmbedding", "from llama_index.embeddings.google_genai import GoogleGenAIEmbedding as GeminiEmbedding")
    if new_content != content:
        with open(filepath, "w") as f: f.write(new_content)

# ---------------------------------------------------------
# 🔥 HOT-PATCH 2: THE ARXIV DIRECT INTERCEPTOR
# ---------------------------------------------------------
from deepcollector.tools.research import ResearchTools

# ✨ ANTI-RECURSION GUARD
if hasattr(ResearchTools, 'tool_load_url') and not getattr(ResearchTools, '_arxiv_patched', False):
    _original_tool_load_url = ResearchTools.tool_load_url

    def patched_tool_load_url(self, url: str, *args, **kwargs):
        if isinstance(url, str) and "arxiv.org" in url:
            match = re.search(r'(\d{4}\.\d{4,5}(v\d+)?|[a-z\-]+/\d{7})', url)
            if match:
                paper_id = match.group(1)
                print(f"\n   📥 [ArXiv Interceptor] Identified {paper_id}. Bypassing web scraper to pull binary PDF...")
                try:
                    import arxiv
                    import pypdf
                    client = arxiv.Client()
                    search = arxiv.Search(id_list=[paper_id])
                    paper = next(client.results(search))
                    target_url = paper.pdf_url.replace("http://", "https://")

                    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
                    response = requests.get(target_url, headers=headers, timeout=30)

                    if response.status_code == 200:
                        text = ""
                        pdf_reader = pypdf.PdfReader(io.BytesIO(response.content))
                        for page in pdf_reader.pages[:50]:
                            text += (page.extract_text() or "") + "\n"

                        if text and len(text.split()) >= 15:
                            print(f"   ✅ Successfully extracted {len(text)} chars from PDF into memory.")
                            return [{"url": url, "content": text, "title": f"ArXiv Paper {paper_id}", "type": "Direct Load"}]
                        else:
                            print(f"   ❌ Failed length check: only extracted {len(text)} chars.")
                    else:
                        print(f"   ❌ Failed to fetch PDF. HTTP Status: {response.status_code}")
                except Exception as e:
                    print(f"   ⚠️ ArXiv extraction error: {e}")

        return _original_tool_load_url(self, url, *args, **kwargs)

    ResearchTools.tool_load_url = patched_tool_load_url
    ResearchTools._arxiv_patched = True
    print("   ✅ ArXiv PDF Interceptor Online.")

# ---------------------------------------------------------
# 🔥 HOT-PATCH 3: FORCE ALL-PRO CLOUD
# ---------------------------------------------------------
import deepcollector.utils.initialization as init_mod
if not getattr(init_mod, '_pro_override_patched', False):
    _original_init_apis = init_mod.initialize_apis
    def patched_init_apis(config):
        keys, Models = _original_init_apis(config)
        print("   🧠 [Architecture Override] Forcing FLASH pool to use PRO models.")
        Models.FLASH_POOL = Models.PRO_POOL
        Models.MODEL_STANDARD = Models.PRO_POOL[0]
        Models.MODEL_RAG_PRIMARY = Models.PRO_POOL[0]
        return keys, Models
    init_mod.initialize_apis = patched_init_apis
    init_mod._pro_override_patched = True

# ---------------------------------------------------------
# EXECUTOR & CONFIGURATION
# ---------------------------------------------------------
os.environ["DEEPCOLLECTOR_USE_VLLM"] = "False"
os.environ["DEEPCOLLECTOR_LLM_BACKEND"] = "GEMINI"
os.environ["DEEPCOLLECTOR_SEARCH_BACKEND"] = "GEMINI"
os.environ["BENCHMARK_MODE"] = "CLOUD"
os.environ["ENABLE_DEEP_RESEARCH_FLAG"] = "True"
os.environ["APPLY_AMNESIA"] = "True"

def get_secret(secret_name, fallback_value=""):
    try: return userdata.get(secret_name) or fallback_value
    except: return fallback_value

print("\n🔑 Authenticating with Google Sheets...")
creds, _ = default()
gc = gspread.authorize(creds)
SECRETS = {"GEMINI_API_KEY": get_secret("GEMINI_API_KEY")}

from deepcollector.config.settings import AppConfig
from deepcollector.core.executor import execute_jobs
from deepcollector.core.agent import CatalogAgent

# ---------------------------------------------------------
# 🔥 AMNESIA & EXPORT PATCHES (With Anti-Recursion Guards)
# ---------------------------------------------------------
if hasattr(CatalogAgent, 'phase_0_bootstrapping') and not getattr(CatalogAgent, '_bootstrap_patched', False):
    _original_bootstrap = CatalogAgent.phase_0_bootstrapping
    def patched_bootstrap(self):
        print("    🛑 [BENCHMARK MODE] Blind Bootstrapping Enabled.")
        old_gspread = getattr(self.config, 'GSPREAD_AVAILABLE', False)
        self.config.GSPREAD_AVAILABLE = False
        res = _original_bootstrap(self)
        self.config.GSPREAD_AVAILABLE = old_gspread
        return res
    CatalogAgent.phase_0_bootstrapping = patched_bootstrap
    CatalogAgent._bootstrap_patched = True

if hasattr(CatalogAgent, 'phase_1a_deep_research') and not getattr(CatalogAgent, '_dr_patched', False):
    _original_phase_1 = CatalogAgent.phase_1a_deep_research
    def patched_phase_1(self):
        _original_phase_1(self)
        print("\n    🛑 [AMNESIA PATCH] Wiping variables to force RAG Extraction...")
        fields_to_wipe = ["Domain", "Detailed Description", "Time interval between points", "Number of Time Points", "Number of Locations/Series", "Variables per Location", "Total Variables", "Comments on Data Preparation"]
        for item in self.state.catalog:
            name = item.get("Dataset Name", {}).get("value")
            if not name or name == "[missing]": continue
            for field in fields_to_wipe:
                if self.state.get_cell_data(name, field).get("value") not in ["[missing]", ""]:
                    self.state.update_cell_data(name, field, {"value": "[missing]", "confidence": 0.0})
    CatalogAgent.phase_1a_deep_research = patched_phase_1
    CatalogAgent._dr_patched = True

if hasattr(CatalogAgent, '_export_run_data') and not getattr(CatalogAgent, '_export_patched', False):
    _original_export = CatalogAgent._export_run_data
    def patched_export(self):
        try: _original_export(self)
        except Exception: pass
        try:
            df = self.get_catalog_report(full_details=True)
            if df is None or df.empty:
                df = pd.DataFrame([{"Dataset Name": "NONE_FOUND", "Completeness (High Conf %)": "0.0%"}])
            proj_name = str(getattr(self.config, 'CURRENT_PROJECT_NAME', 'UNKNOWN'))

            # 🔥 CHANGED: Labeled as Pro-Monolithic for Matrix parity
            safe_model = "Gemini-Cloud-PRO-Monolithic"

            df.insert(0, "Project", proj_name)
            df.insert(1, "Benchmark_Model", safe_model)
            safe_proj_name = re.sub(r'[^A-Za-z0-9_\-]', '_', proj_name).strip('_')
            timestamp = time.strftime('%Y%m%d_%H%M')
            out_path = f"/content/drive/MyDrive/Bench_{safe_proj_name}_{safe_model}_{timestamp}.csv"
            df.to_csv(out_path, index=False)
            print(f"🎉 [Export] Benchmark Data saved to Colab Drive: {out_path}")
        except Exception as e:
            print(f"❌ Benchmark CSV Override Failed: {e}")
    CatalogAgent._export_run_data = patched_export
    CatalogAgent._export_patched = True

config = AppConfig(
    VERBOSITY_LEVEL=1, SECRETS=SECRETS,
    GOOGLE_SHEET_KB_INPUT = get_secret("KB_SHEET_ID"),
    GOOGLE_SHEET_PROJECT_LIST_INPUT = get_secret("PROJECT_LIST_ID"),
    GOOGLE_DRIVE_SHEET_FOLDER_ID = get_secret("DRIVE_SHEET_FOLDER_ID"),
    GOOGLE_DRIVE_LOG_FOLDER_ID = get_secret("DRIVE_LOG_FOLDER_ID"),
    ENABLE_DEEP_RESEARCH=True,
    ENABLE_GOLDEN_FASTPATH=False,
    ENABLE_PREFLIGHT_CRAWLER=True, ENABLE_ARBITRATION_PROMPT=True, ENABLE_STRICT_TAXONOMY=True,
    ENABLE_MULTI_QUERY_RAG=True, ENABLE_VARIANT_MAPPING=True, ENABLE_SINGLETON_VERIFICATION=True, ENABLE_ORACLE_SEARCH=True
)

# 🔥 RATE LIMIT PREVENTION
# The Pro models have tighter limits than Flash. If we encounter Event Loop stops, lowering this prevents the crash.
config.CELLULAR_RAG_BATCH_SIZE = 10

os.environ["DEEPCOLLECTOR_SHEET_FOLDER_ID"] = config.GOOGLE_DRIVE_SHEET_FOLDER_ID or ""
os.environ["DEEPCOLLECTOR_LOG_FOLDER_ID"] = config.GOOGLE_DRIVE_LOG_FOLDER_ID or ""
config.recalculate_runtime_parameters()
config._process_sheet_ids()

# ==========================================
# 🎯 EXECUTE THE RUN
# ==========================================
PROJECT_NAMES = ["TimeBench", "LOTSA", "TEMPO", "TSFM", "KaggleTS", "M2", "M6"]

print(f"\n🚀 Firing TRUE CLOUD BASELINE for {PROJECT_NAMES}...")
%cd /content/DeepKG
try:
    execute_jobs(mode="AGENT", project_names=PROJECT_NAMES, base_config=config, gc_client=gc, dry_run=False)
except Exception as e:
    print(f"❌ Main Execution Failed: {e}")

<IPython.core.display.Javascript object>

🔧 Patching DeepCollector for the new google-genai SDK...
✅ deepcollector/utils/profiler.py written.
   ✅ ArXiv PDF Interceptor Online.

🔑 Authenticating with Google Sheets...
✅ deepcollector/config/schema.py written (V83: URL Separation).
✅ deepcollector/config/schema.py written (V83: URL Separation).
✅ deepcollector/core/state.py written (Added Search Memory Set).
✅ deepcollector/tools/ddi.py written.
✅ deepcollector/kb/manager.py written (100% Full Un-Truncated + GSpread V5/V6 Safe Fetch).
✅ deepcollector/core/rag_engine.py written (100% Fully Expanded + Native JSON Prompt Fix).
✅ deepcollector/harvesting/base_harvester.py written (HDK Compatible).
✅ deepcollector/harvesting/uci_harvester.py written (V11.2: Deep Think Fix: Single Line Enforcer).
✅ deepcollector/kb/maintenance.py written (SyntaxWarning Fix).
✅ deepcollector/tools/deep_research_runner.py written (Configurable Agent Model).
✅ deepcollector/utils/analytics.py written (V8: Workload Aggregation Fix).
✅ deepcollector/core/a

    🔍 Verifying Pro (Reasoning) Pool...
       ✅ Verified 4 models: ['gemini-3.1-pro-preview', 'gemini-3-pro-preview', 'gemini-2.5-pro', 'gemini-pro-latest']
    🔍 Verifying Flash (Extraction) Pool...
       ✅ Verified 5 models: ['gemini-3.5-flash', 'gemini-3-flash-preview', 'gemini-2.5-flash', 'gemini-2.0-flash', 'gemini-2.0-flash-lite']
    ✅ Gemini Client initialized.
    🎭 Active Roles:
       - Planner/Search: gemini-3.1-pro-preview
       - RAG Extraction: gemini-3.5-flash
   🧠 [Architecture Override] Forcing FLASH pool to use PRO models.
🔧 Configuring LlamaIndex (Native Interface)...


    🔍 Initializing Native Embedding Model (Cascade)...


      ✅ Selected: models/gemini-embedding-001 (via google.genai)
    ✅ LlamaIndex Configured Successfully.

🚀 BATCH PROGRESS: [1/7] Processing Project: TimeBench
🔄 Loading Project Context for TimeBench...
🔄 [ProjectLoader] Dynamically loading configuration for ID: PROJ_TIMEBENCH...
🌐 [ExternalKnowledge] Initialization complete. Ready to load sources.
💡 [ExternalKnowledge] Loading Canonical Projects from Sheet ID: 1gJ6oHZj...rFwE
    ✅ [ExternalKnowledge] Loaded 103 records.
    ✅ [ProjectLoader] Configuration loaded. Method: 1, URLs: 1.
    🔗 Source URL [1]: https://github.com/thuml/Sundial
    ✅ Context Loaded: 'TIMEBENCH: PROJ_TimeBench...'

🚀 Starting Job JOB_67607B (AGENT) for: 'TIMEBENCH: PROJ_TimeBench'
📝 Job Comment: TimeBench Batch AGENT execution.
    🛑 [BENCHMARK MODE] Blind Bootstrapping Enabled.

=== PHASE 0: BOOTSTRAPPING ===
🌐 Loading 1 initial URLs...
    ✅ [Vector Index] Inserted 1 docs (2 nodes).


...

    ✅ [BM25 Index] Rebuilt with 399 nodes.
🛠️ Executing Targeted Search: how many variables features dimensions in 'submissions.csv' dataset
🌐 [Tool: Search/Fetch] Query: 'how many variables features dimensions in 'submissions.csv' dataset'
    ✅ Gemini Search returned 7 usable results.
    ✅ [Vector Index] Inserted 5 docs (5 nodes).


DEBUG:bm25s:Building index from IDs objects


    ✅ [BM25 Index] Rebuilt with 404 nodes.
🛑 [Early Termination] Extraction stalled. Stopping.

=== PHASE 3.5: MAINTENANCE & REPAIR (V166) ===
    🧹 [Doctor] Standardizing names...
    🧹 [Doctor] Cleaning URL formatting and purging banned domains...
    🧠 [Doctor] Running STRICT Exact Deduplication on 3 items...
    ✅ No exact duplicates found.
    🩺 [Doctor] Scanning 3 items for missing or redirect URLs...
    🧟 Zombie / Paper URL Found (Needs real URL): assets_m6.csv
🌐 [Tool: Search/Fetch] Query: 'official download url repository github dataset 'assets_m6.csv''
    ✅ Gemini Search returned 4 usable results.
      ✨ Healed with Real URL: https://raw.githubusercontent.com/Mcompetitions/M6-methods/main/assets_m6.csv`
    🧟 Zombie / Paper URL Found (Needs real URL): submissions.csv
🌐 [Tool: Search/Fetch] Query: 'official download url repository github dataset 'submissions.csv''
    ✅ Gemini Search returned 2 usable results.
      ✨ Healed with Real URL: https://github.com/IBM/Project_Cod

    ✅ Success! CSV cleanly uploaded to Drive with ID: 1ncAbMVFBBehnHoRuM7B8rwXotKeqL4QD
🎉 [Export] Benchmark Data saved to Colab Drive: /content/drive/MyDrive/Bench_M6_Gemini-Cloud-PRO-Monolithic_20260703_0019.csv

💰 VALUE ANALYSIS REPORT (Tri-State Updates)
Phase                             Time (s)  Time %      Net Added    Deletions    Filled    Refined    Confirmed    Total Cell Ops    Velocity
------------------------------  ----------  --------  -----------  -----------  --------  ---------  -----------  ----------------  ----------
Bootstrapping                         37    1.7%                0            0         0          0            0                 0        0
Phase 1a: Deep Research              201.5  9.0%                1            0         8          0            0                 8        0.3
Phase 1b: Standard Discovery          23.1  1.0%                2            0         4          0            0                 4        5.21
Phase 1.5: Golden KB Fast-Path


☁️ Uploading Console Log (TimeBench_LOTSA_TEMPO_TSFM_Kag_20260702_1433_ConsoleLog.txt) to Google Drive...
✅ Success! Log File uploaded to Drive with ID: 1c7-JyntEA9O8_9g8A5kISOdID2fCzhNL
